# Data Project

Import and set magics:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# autoreload modules when code is run
%load_ext autoreload
%autoreload 2

# user written modules
import dataproject

#Importing APIs
from fredapi import Fred
from dstapi import DstApi


%pip install git+https://github.com/alemartinello/dstapi
%pip install fredapi

In [ ]:
data = dataproject.load_data()
data = dataproject.process_data(data)

## 1. Aggregate inflation in Denmark

**1.1 Basics**

In [ ]:
#we load the data from DST:
PRIS113 = DstApi('PRIS113')
PRIS113.tablesummary(language='en')

Clean data:

In [ ]:
#a. we set the parameters
params = {
    'table': 'PRIS113',
    'format': 'BULK', 
    'lang': 'en',
    'variables': [
        {'code': 'TYPE', 'values': ['INDEKS']},
        {'code': 'Tid', 'values': ['*']},
        ]
    }

# b. download the table
CPI = PRIS113.get_data(params=params)

#c. set table structure
CPI['date'] = pd.to_datetime(CPI['TID'], format='%YM%m')
CPI['INDHOLD'] = CPI['INDHOLD'].astype(float)
CPI = CPI.drop(columns=['TYPE', 'TID'])
CPI = CPI.rename(columns={'INDHOLD': 'index'})
CPI = CPI.set_index('date').sort_index()

# d. display
display(CPI.head())
display(CPI.info())

1. Figure showing CPI with indexing 2020=100

In [ ]:
#a. we index to 100 on average in 2020
CPI['index'] = CPI['index']/CPI.loc[CPI.index.year == 2020, 'index'].mean() * 100

#b. we plot CPI with index 2020=100
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(CPI.index, CPI['index'], label='CPI', color='tab:blue')
ax.set_title('Consumer Price Index — 2020 average = 100')
ax.set_xlabel('Year')
ax.set_ylabel('Index (2020 = 100)')
ax.grid(axis='y', linestyle=':', linewidth=0.6)

#c. we display CPI
display(CPI.head())
plt.show()

2. Figure showing month to month inflation rate

In [ ]:
#We add a coloumn with the month-to-month inflation rate
CPI['monthly_inflation'] = CPI['index'].pct_change()*100

# we plot the month to month inflation rate
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(CPI.index, CPI['monthly_inflation'], label='CPI', color='tab:red')
ax.set_title('Month-to-month inflation rate for CPI')
ax.set_xlabel('Year')
ax.set_ylabel('Inflation rate (%)')
ax.grid(axis='y', linestyle=':', linewidth=0.6)  

3. Figure showing 12-month inflation rate

In [ ]:
#We add a coloumn with the 12-month inflation rate
CPI['yearly_inflation'] = CPI['index'].pct_change(12, fill_method=None)*100

# we plot the month to month inflation rate
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(CPI.index, CPI['yearly_inflation'], label='CPI', color='tab:red')
ax.set_title('12-month inflation rate for CPI')
ax.set_xlabel('Year')
ax.set_ylabel('Inflation rate (%)')
ax.grid(axis='y', linestyle=':', linewidth=0.6)  

1.1 Conlusion: When did the post-pandemic inflation surge end in Denmark?

In [ ]:
#we display the inflation rate for the months between January 2020 and June 2024
display(CPI.loc['2020-01-01':'2024-06-01', ['yearly_inflation']])



Based on the table above, the inflation levels were notably high during 2022. However, they stabilize around may 2023. 

**1.2 Instantaneous inflation**

Instantaneous inflation rate is defined as

$$
\pi_{t}^{12,\alpha} =\left(\Pi_{k=0}^{11}\left(1+\pi_{t-k}\right)^{\kappa\left(k,\alpha\right)}\right)-1\\
\kappa\left(k,\alpha\right) =\frac{\left(T-k\right)^{\alpha}}{\sum\left(T-k\right)^{\alpha}}T
$$

1. plot $$\kappa\left(k,\alpha\right)$$ 

In [ ]:
#a. vi definerer funktionen:
def kappa(k,alpha, T=12):
    numerator = (T - k)**alpha
    denominator = np.sum((T-j)**alpha for j in range(T))
    return numerator / denominator * T

#b. vi plotter kappa for alpha=[0,1,2,3] og k=[0,1,2,...,11]
T = 12
ks = np.arange(T)
alphas = [0,1,2,3]

for alpha in alphas:
    kappas = [kappa(k, alpha, T) for k in ks]
    plt.plot(ks, kappas, label=f'$\\alpha={alpha}$')
plt.xlabel(r'$k$ (months ago)')
plt.ylabel(r'$\kappa(k,\alpha)$')
plt.title(r'$\kappa(k,\alpha)$ for different values of $\alpha$')
plt.legend()
plt.show()

## Question 2

We process the data by ...

In [ ]:
data = dataproject.process_data(data)

In [ ]:
fig = plt.figure(figsize=(7,4))
ax = fig.add_subplot(111)

ax.plot(data['log_GDP'],label='log(GDP)')

ax.set_title('Log GDP over time')
ax.set_xlabel('Year')
ax.set_ylabel('Log GDP')

ax.legend();

We find that that ...